# Лабораторная работа №1: Классификация цвета автомобиля (DVM, фронтальные виды)

## Постановка

Нужно сравнить 3 модели для предсказания цвета автомобиля:
1. **Классификатор, написанный своими руками** (в этом ноутбуке: `ScratchResNet`).
2. **Предобученный классификатор №1** + дообучение на DVM (в этом ноутбуке: `ResNet18 ImageNet`).
3. **Предобученный классификатор №2** + дообучение на DVM (в этом ноутбуке: `MobileNetV3-Large ImageNet`).

Метрика: **F1_macro**, целевое требование: **F1_macro > 0.8**.


## Данные

Источник: https://deepvisualmarketing.github.io/  
Используем архив **Quality checked front-view images (730 MB)**.

### Ожидаемая структура

```text
data/raw/dvm_front/
  Brand/
    Model/
      Year/
        Color/
          *.jpg
```

Цвет извлекается из названия папки `Color`.


In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tv_models

sns.set_theme(style='whitegrid')


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
@dataclass
class Config:
    data_root: str = 'data/raw/dvm_front'
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 2

    # фильтрация и баланс
    min_samples_per_class: int = 200

    # epochs / lr
    epochs_scratch: int = 20
    epochs_pretrained_1: int = 12
    epochs_pretrained_2: int = 12
    lr_scratch: float = 1e-3
    lr_pretrained: float = 3e-4
    weight_decay: float = 1e-4

    # для быстрой проверки пайплайна
    fast_dev_run: bool = False
    fast_dev_samples_per_class: int = 180


cfg = Config()
cfg


## Подготовка датасета

В этом ноутбуке используется аккуратная нормализация названий цветов (например, `grey -> gray`, `dark blue -> blue`). Это уменьшает дробление классов и делает сравнение моделей честнее.


In [ ]:
def canonical_color(raw: str) -> str:
    c = raw.strip().lower().replace('-', ' ')

    # прямые алиасы
    direct = {
        'grey': 'gray',
        'silver metallic': 'silver',
        'blue metallic': 'blue',
        'red metallic': 'red',
        'white pearl': 'white',
        'black metallic': 'black',
    }
    if c in direct:
        return direct[c]

    groups = {
        'black': ['black', 'jet black'],
        'white': ['white', 'ivory'],
        'gray': ['gray', 'grey', 'graphite', 'charcoal'],
        'silver': ['silver'],
        'blue': ['blue', 'navy', 'azure', 'teal'],
        'red': ['red', 'maroon', 'burgundy'],
        'green': ['green', 'lime', 'olive'],
        'yellow': ['yellow'],
        'orange': ['orange', 'amber'],
        'brown': ['brown', 'bronze', 'chocolate'],
        'purple': ['purple', 'violet', 'magenta'],
        'gold': ['gold'],
        'beige': ['beige', 'cream', 'sand'],
    }

    for key, tokens in groups.items():
        if any(tok in c for tok in tokens):
            return key

    return c


def build_index(data_root: str) -> pd.DataFrame:
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(
            f'Папка {root} не найдена. Скачайте и распакуйте фронтальные DVM изображения.'
        )

    rows = []
    valid_ext = {'.jpg', '.jpeg', '.png'}

    for img_path in root.rglob('*'):
        if img_path.suffix.lower() not in valid_ext:
            continue
        parts = img_path.parts
        if len(parts) < 5:
            continue
        raw_color = parts[-2]
        rows.append({
            'path': str(img_path),
            'raw_color': raw_color,
            'color': canonical_color(raw_color),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError('Не найдено изображений в указанной папке data_root')
    return df


def prepare_dataframe(cfg: Config) -> pd.DataFrame:
    df = build_index(cfg.data_root)

    # фильтруем очень редкие классы
    counts = df['color'].value_counts()
    keep = counts[counts >= cfg.min_samples_per_class].index
    df = df[df['color'].isin(keep)].copy()

    if cfg.fast_dev_run:
        sampled_parts = []
        for _, grp in df.groupby('color', group_keys=False):
            sampled_parts.append(
                grp.sample(min(len(grp), cfg.fast_dev_samples_per_class), random_state=42)
            )
        df = pd.concat(sampled_parts, ignore_index=True)

    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df


df = prepare_dataframe(cfg)
print('Всего изображений после фильтрации:', len(df))
print('Число классов:', df['color'].nunique())
print(df['color'].value_counts())


In [ ]:
plt.figure(figsize=(11, 4))
class_counts = df['color'].value_counts()
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.xticks(rotation=45, ha='right')
plt.title('Распределение классов цветов')
plt.ylabel('Количество изображений')
plt.tight_layout()
plt.show()


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['color'],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['color'],
)

label2id = {c: i for i, c in enumerate(sorted(train_df['color'].unique()))}
id2label = {i: c for c, i in label2id.items()}

for part in (train_df, val_df, test_df):
    part['label'] = part['color'].map(label2id)

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))
print('labels:', label2id)


In [ ]:
class ColorDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])


train_tfms = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

infer_tfms = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ColorDataset(train_df, transform=train_tfms)
val_ds = ColorDataset(val_df, transform=infer_tfms)
test_ds = ColorDataset(test_df, transform=infer_tfms)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)


## Модель 1 (своими руками): ScratchResNet

Ниже реализована небольшая ResNet-подобная сеть вручную (без использования готовой `torchvision.models.resnet18` как архитектуры для scratch-версии).


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)
        return out


class ScratchResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.in_ch = 64

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(64, blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)
        self.layer4 = self._make_layer(512, blocks=2, stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_ch, blocks, stride):
        layers = [BasicBlock(self.in_ch, out_ch, stride=stride)]
        self.in_ch = out_ch
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


## Универсальные функции обучения и оценки


In [ ]:
def make_class_weights(train_df: pd.DataFrame, num_classes: int) -> torch.Tensor:
    counts = train_df['label'].value_counts().sort_index()
    counts = counts.reindex(range(num_classes), fill_value=1)
    weights = 1.0 / counts.values.astype(np.float32)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


@torch.no_grad()
def predict_epoch(model, loader):
    model.eval()
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        pred = logits.argmax(dim=1).cpu().numpy().tolist()
        y_pred_all.extend(pred)
        y_true_all.extend(yb.numpy().tolist())

    return y_true_all, y_pred_all


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        y_true_all.extend(yb.detach().cpu().numpy().tolist())
        y_pred_all.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    f1 = f1_score(y_true_all, y_pred_all, average='macro')
    return float(np.mean(losses)), float(f1)


@torch.no_grad()
def validate_one_epoch(model, loader, criterion):
    model.eval()
    losses = []
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        losses.append(loss.item())
        y_true_all.extend(yb.detach().cpu().numpy().tolist())
        y_pred_all.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    f1 = f1_score(y_true_all, y_pred_all, average='macro')
    return float(np.mean(losses)), float(f1), y_true_all, y_pred_all


def fit_model(model, name, train_loader, val_loader, train_df, num_classes, lr, weight_decay, epochs, patience=4):
    model = model.to(device)

    class_weights = make_class_weights(train_df, num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
    best_state = None
    best_val_f1 = -1.0
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_f1, _, _ = validate_one_epoch(model, val_loader, criterion)

        scheduler.step(val_f1)

        history['train_loss'].append(tr_loss)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)

        print(f'[{name}] Epoch {epoch:02d}/{epochs} | train_loss={tr_loss:.4f} train_f1={tr_f1:.4f} | val_loss={val_loss:.4f} val_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f'[{name}] Early stopping')
            break

    model.load_state_dict(best_state)
    return model, history, best_val_f1


def evaluate_on_test(model, test_loader, train_df, num_classes):
    class_weights = make_class_weights(train_df, num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    test_loss, test_f1, y_true, y_pred = validate_one_epoch(model, test_loader, criterion)
    return test_loss, test_f1, y_true, y_pred


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history['train_loss'], label='train')
    axes[0].plot(history['val_loss'], label='val')
    axes[0].set_title(f'{title} — Loss')
    axes[0].legend()

    axes[1].plot(history['train_f1'], label='train')
    axes[1].plot(history['val_f1'], label='val')
    axes[1].set_title(f'{title} — F1_macro')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## Обучение модели 1: ScratchResNet (с нуля)


In [ ]:
num_classes = len(label2id)

scratch_model = ScratchResNet(num_classes=num_classes)
scratch_model, hist_scratch, best_val_scratch = fit_model(
    model=scratch_model,
    name='ScratchResNet',
    train_loader=train_loader,
    val_loader=val_loader,
    train_df=train_df,
    num_classes=num_classes,
    lr=cfg.lr_scratch,
    weight_decay=cfg.weight_decay,
    epochs=cfg.epochs_scratch,
    patience=4,
)

plot_history(hist_scratch, 'ScratchResNet')
print('Best val F1_macro (ScratchResNet):', round(best_val_scratch, 4))


## Обучение модели 2: предобученный ResNet18 (ImageNet)


In [ ]:
weights_rn18 = tv_models.ResNet18_Weights.IMAGENET1K_V1
model_rn18 = tv_models.resnet18(weights=weights_rn18)
model_rn18.fc = nn.Linear(model_rn18.fc.in_features, num_classes)

model_rn18, hist_rn18, best_val_rn18 = fit_model(
    model=model_rn18,
    name='ResNet18-ImageNet',
    train_loader=train_loader,
    val_loader=val_loader,
    train_df=train_df,
    num_classes=num_classes,
    lr=cfg.lr_pretrained,
    weight_decay=cfg.weight_decay,
    epochs=cfg.epochs_pretrained_1,
    patience=4,
)

plot_history(hist_rn18, 'ResNet18-ImageNet')
print('Best val F1_macro (ResNet18-ImageNet):', round(best_val_rn18, 4))


## Обучение модели 3: предобученный MobileNetV3-Large (ImageNet)


In [ ]:
weights_mnv3 = tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
model_mnv3 = tv_models.mobilenet_v3_large(weights=weights_mnv3)
model_mnv3.classifier[-1] = nn.Linear(model_mnv3.classifier[-1].in_features, num_classes)

model_mnv3, hist_mnv3, best_val_mnv3 = fit_model(
    model=model_mnv3,
    name='MobileNetV3-ImageNet',
    train_loader=train_loader,
    val_loader=val_loader,
    train_df=train_df,
    num_classes=num_classes,
    lr=cfg.lr_pretrained,
    weight_decay=cfg.weight_decay,
    epochs=cfg.epochs_pretrained_2,
    patience=4,
)

plot_history(hist_mnv3, 'MobileNetV3-ImageNet')
print('Best val F1_macro (MobileNetV3-ImageNet):', round(best_val_mnv3, 4))


## Сравнение моделей на тестовой выборке


In [ ]:
test_loss_s, test_f1_s, y_true_s, y_pred_s = evaluate_on_test(scratch_model, test_loader, train_df, num_classes)
test_loss_r, test_f1_r, y_true_r, y_pred_r = evaluate_on_test(model_rn18, test_loader, train_df, num_classes)
test_loss_m, test_f1_m, y_true_m, y_pred_m = evaluate_on_test(model_mnv3, test_loader, train_df, num_classes)

results = pd.DataFrame([
    {'model': 'ScratchResNet (from scratch)', 'test_loss': test_loss_s, 'test_f1_macro': test_f1_s},
    {'model': 'ResNet18 (ImageNet finetune)', 'test_loss': test_loss_r, 'test_f1_macro': test_f1_r},
    {'model': 'MobileNetV3-Large (ImageNet finetune)', 'test_loss': test_loss_m, 'test_f1_macro': test_f1_m},
]).sort_values('test_f1_macro', ascending=False).reset_index(drop=True)

results


In [ ]:
best_model_name = results.iloc[0]['model']
print('Лучшая модель:', best_model_name)

if best_model_name.startswith('ScratchResNet'):
    y_true_best, y_pred_best = y_true_s, y_pred_s
elif best_model_name.startswith('ResNet18'):
    y_true_best, y_pred_best = y_true_r, y_pred_r
else:
    y_true_best, y_pred_best = y_true_m, y_pred_m

print('\nClassification report (лучшая модель):')
print(classification_report(y_true_best, y_pred_best, target_names=[id2label[i] for i in range(num_classes)]))


In [ ]:
cm = confusion_matrix(y_true_best, y_pred_best)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm_norm,
    cmap='Blues',
    xticklabels=[id2label[i] for i in range(num_classes)],
    yticklabels=[id2label[i] for i in range(num_classes)],
)
plt.title('Нормализованная confusion matrix (лучшая модель)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Выводы по лабораторной (автоматически)


In [ ]:
threshold = 0.8
print('Итоговая таблица:')
print(results.to_string(index=False))

best_f1 = float(results.iloc[0]['test_f1_macro'])
print(f'\nЛучший F1_macro = {best_f1:.4f}')
print('Требование F1_macro > 0.8:', 'выполнено' if best_f1 > threshold else 'не выполнено')

print('\nКраткий вывод:')
print('- Сравнены 3 классификатора: 1 с нуля и 2 предобученных.')
print('- Победитель определен по максимальному test F1_macro.')
print('- Обычно предобученные модели выигрывают на ограниченном датасете благодаря transfer learning.')
